# IHC Global Analysis — Part 2: Data aggregation and event summaries

## What this notebook does 

This notebook is the **final step** of the IHC global analysis workflow.
Run it **after** peak detection has already been completed (with `traceExplorer`).

In simple terms, this notebook takes the detected calcium peaks and turns them into summary tables that are ready for statistics and plotting.

### Main steps

1. **Load and clean data**
   - Reads the input tables and traces.
   - Keeps only IHC cells and organizes metadata (mouse, age, recording, etc.).

2. **Measure peak features**
   - Calculates values such as peak timing, width, amplitude, and frequency per cell.
   - Marks peaks as high quality using quality rules in the code.

3. **Build event-level summaries**
   - Groups peaks that happen close in time into shared events.
   - Current rule: peaks within about **2 seconds** are treated as the same event.
   - Events can be split if responding cells are too far apart in the cell order (current threshold in code: `maxSkip = 4`).

4. **Create final output tables**
   - `master`: one row per cell (cell-level summary; useful for cell/recording/mouse frequencies).
   - `masterIndependentCombined`: equivalent to `master`, but cells are tracked across multiple recordings in the same field of view.
   - `masterEventsHq`: one row per high-quality event (event-level summary; includes how many cells respond).
   - `masterPeak`: one row per peak (peak-level summary; useful for distributions like peak width).
   - `masterPeakHq`: high-quality subset of `masterPeak`.

### Practical note

You can think of each table as a spreadsheet at a different level:
- **Cell level** (`master`)
- **Event level** (`masterEventsHq`)
- **Single-peak level** (`masterPeak`)

## Setup and input files

The next cells import required libraries, load metadata, and read previously generated analysis files (`.xlsx` and `.parquet`).

If file paths are different on your machine, update `fileHeader`, `resultsFOlder`, and `localDrive` before running.

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%pylab
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import numpy as np
import os
import sys
sys.path.append('../../src')
# os.environ['R_HOME'] = r"C:\\Users\\LabAdmin\\anaconda3\\envs\\napari\\Lib\\R"
# %load_ext rpy2.ipython
import seaborn as sns
import visualisationTools as vu
import traceUtilities as tu
from traceUtilities import fillMissingValues,  rollingMedianCorrection, determineLocalDrive
import ast 
from movieTools import determineCellTypes

parameterFolder = '../../parameters/'
fileHeader = [
'alpha9'
]
alldatalist  = []
for h in fileHeader:
    inputFilename = os.path.join('../../',h)+'.xlsx'

    alldata1 = pd.read_excel(inputFilename)
    alldatalist.append(alldata1[alldata1['discard']!=1])

alldata = pd.concat(alldatalist, ignore_index=True)
alldata = alldata[~alldata['Folder'].isna()]
alldata = alldata.reset_index()
alldata['Mouse ID'] = alldata['Strain'] + alldata['Mouse ID'].astype(str)

alldata = alldata.sort_values(['Age','Mouse ID','Folder']).reset_index()


resultsFOlder = '../../analysis_results/'
fileHeader = 'alpha9'

inputFilename = os.path.join(resultsFOlder,fileHeader)+'.xlsx'

tempFilename = '../../analysis_results/master_temp.csv'

# Change drive automatically by probing candidate locations (Z: is fallback)
localDrive = determineLocalDrive(
    alldata,
    folderColumn='Folder',
    candidates=['/media/marcotti-lab', 'D:', 'E:', 'F:', 'Z:'],
    nRandomChecks=3,
    randomState=0,
 )
print(f'Using localDrive: {localDrive}')

alldata['Folder'] = localDrive + alldata['Folder'].str[2:].str.replace('\\','/')

In [ ]:
#Load previous result
master  =  pd.read_excel(inputFilename) #pd.read_csv(tempFilename)#
#KEEP ONLY IHCS
master = master[master['Cell type']==1]
master['Peak positions'] = master['Peak positions'].apply(ast.literal_eval)
master['Independent recordings number'] = master['Independent recordings number'].astype(object)

alltraces = pd.read_parquet(os.path.join(resultsFOlder,f'{fileHeader}_events_alltraces.parquet'))


allCorrTraces = pd.read_parquet(os.path.join(resultsFOlder,f'{fileHeader}_events_allCorrTraces.parquet'))


# allspikeprob = pd.read_csv(os.path.join(resultsFOlder,'alpha9_event_alltraces_cascadespikeprob_rollMedCorr1000.csv')) # these are the spike probabilities estimated with cparquetade


## Optional visual inspection of concatenated traces

These cells concatenate traces from matched recordings and plot them as a quick visual check before event-level analysis.

In [ ]:
#Generate a dictionary with all the concatenated traces from independent recordings

id_table = (
    master[['Independent recordings ID', 'Age', 'Mouse ID', 'Strain']]
    .drop_duplicates()
    .sort_values(['Age', 'Mouse ID', 'Strain'], kind='mergesort')
)
identifiers = id_table['Independent recordings ID'].tolist()

alldff0s  = {}
for x in identifiers:
        el = master[master['Independent recordings ID']==x]
        el = el[el['Cell type']==1]
        alldff0s[x]= tu.concatenateRecordings(el,alltraces,preprocessing='rolling_percentile')

ages_dict = {}
for x in identifiers:
        ages_dict[x] = id_table[id_table['Independent recordings ID']==x]['Age'].values[0]

In [ ]:
vu.simpleTracePlotter(alltraces=alldff0s,xScaleFactor=60, xlabel='Time (min)',ages_dict=ages_dict)

## Data quality check: trace length consistency

Before calculating event statistics, we verify that trace durations in the saved dataset match the original `traces.csv` files.

Expected result: both printed sets are empty. If not, those recordings may need to be reprocessed.

In [ ]:
def extractTraces(folder,fps = 1,returndFF0 = True,returnCorrelations = True, fillMissingValue = False):
    filename = os.path.join(folder,'processedMovies','traces.csv')
    traces = pd.read_csv(filename,index_col=0)
    x = traces['Frame']/fps
    traces = traces.drop('Frame',axis=1)
    if fillMissingValue:
        traces = fillMissingValues(traces)
    stop = int(traces.shape[0]/2)
    f0 = traces.iloc[:stop,:].quantile(0.3)
    
    dff0 = (traces - f0)/f0



    if returnCorrelations:
        filename = os.path.join(folder,'processedMovies','traces_localCorr.csv')
        correlations = pd.read_csv(filename,index_col=0)
        if returndFF0:
            return x,dff0,correlations
        else:
            return x,traces,correlations

    else:
        if returndFF0:
            return x,dff0
        else:
            return x,traces
        
affectedFolders = []
affectedFolders2 = []
for folder in master['Folder'].unique(): 
    el = master.query('Folder==@folder')
    x,t = extractTraces(folder,el['fps'].values[0], returnCorrelations=False, fillMissingValue= True)
    for i,row in el.iterrows():
        analysedDuration = alltraces[row['Cell ID']].dropna().shape[0]
        realDuration = t[row['RoiN']].shape[0]
        if realDuration>analysedDuration:
            affectedFolders.append(folder)
        if realDuration<analysedDuration:
            affectedFolders2.append(folder)

In [ ]:
# This set should be empty. 
print(set(affectedFolders2))
print(set(affectedFolders))

## Add per-cell peak metadata

This section prepares cell-level information needed for event analysis:
- sequential cell order within each recording
- peak timing and widths
- noise estimates and frequencies

In [ ]:
#Add a sequential number for the ihcs in each recording
for folder in master['Folder'].unique():
    el = master[(master['Cell type'] == 1) & (master['Folder']==folder)]
    cellnumbers = sort([int(r.split('_')[-1]) for r in el['RoiN']])
    cellSequentialNumbers = [i for i in range(el.shape[0])]
    cellDict = dict(zip(cellnumbers,cellSequentialNumbers))
    for j, el in el.iterrows():
        cellN = int(el['RoiN'].split('_')[-1])
        cellSeqN =cellDict[cellN]
        master.loc[j,'Cell sequential number'] = cellSeqN
    

### Current HQ filter behavior (important)

The code below computes high-quality (HQ) labels, but at the moment it keeps all detected peaks (`keepPeak = True`).

So HQ-specific outputs are currently equivalent to using all peaks unless you change the filtering rule.

In [ ]:
from scipy.signal import peak_widths

master['Peak positions (s)'] = None
master['Peak ISI (s)'] = None
master['Peak amplitudes'] = None

master['Peaks half widths (s)'] = None
master['Peaks left limit']  = None
master['Peaks right limit'] = None

master['Average frequency from cascade (Hz)'] = None


#Define Age groups
master.loc[master['Age']<7,'Age group'] = 'First week'
master.loc[master['Age']>=7,'Age group'] = 'Second week'
master.loc[master['Age']>=12,'Age group'] = 'Posthearing'
# Assign a unique Id for each cell in different recordings
master['Unique cell ID'] = master['Independent recordings ID'] + '_' +master['Matched RoiN'].astype(str) 

# Calculate peak widths, limits and fraction of active time. Reorder peaks 
for j,el in master.iterrows():
    
    trace = alltraces[el['Cell ID']].dropna()
    #spikeprob = allspikeprob[el['Cell ID']].dropna()
   # durationSpikeProb = spikeprob.shape[0]/el['fps'] # it can be different from the total duration of the trace due to cutoff at the edges
    peaks = el['Peak positions']
    if not isinstance(peaks, list):
        if isinstance(peaks, np.ndarray):
            peaks = [peaks.item()] if peaks.ndim == 0 else peaks.tolist()
        elif pd.isna(peaks):
            peaks = []
        elif isinstance(peaks, (tuple, set)):
            peaks = list(peaks)
        else:
            peaks = [peaks]

    peaks = sort(peaks)

    master.at[j,'Peak positions'] = peaks

    #Detrend the trace as in the  automatic detection fase to correctly get the original trace the peaks were calculated on
    detect_trace = tu.detrend_z_score(trace, rollingN=2000, savgol_order=21)    

    #Realign peaks on the detect_trace, so that manually selected peaks are aligned to a local maximum.
    window = 10
    peaks2 = []
    for i,p in enumerate(peaks):
        local_window = detect_trace[max(0,p-window):min(len(detect_trace),p+window+1)]
        if len(local_window)>0:
            local_peak = local_window.argmax() + max(0,p-window)
            peaks2.append(local_peak)
        else:
            peaks2.append(p)

    results_full = peak_widths(detect_trace,peaks2,rel_height=0.5)
    master.at[j,'Peaks half widths (s)']  = results_full[0]/el['fps']
    master.at[j,'Peaks left limit']  = results_full[2]
    master.at[j,'Peaks right limit']  = results_full[3]        
    total_duration = size(trace)
    timepoints = set(np.arange(total_duration))   
    for i in range(len(results_full[2])):
        peak_interval = set(np.arange(round(results_full[2][i]),round(results_full[3][i])))
        timepoints = timepoints - peak_interval
    master.loc[j,'Fraction of active time (%)'] = (1 - len(timepoints)/total_duration) *100
    traceCorr = rollingMedianCorrection(trace)
    master.loc[j,'Noise STD'] = traceCorr[list(timepoints)[0:500]].std()#trace[sort(random.choice(list(timepoints),500,replace=False))].std()
#Calculate frequency, ISI, etc..

    if el['Cell type'] == 1:
        peaks = el['Peak positions']
        if not isinstance(peaks, list):
            if isinstance(peaks, np.ndarray):
                peaks = [peaks.item()] if peaks.ndim == 0 else peaks.tolist()
            elif pd.isna(peaks):
                peaks = []
            elif isinstance(peaks, (tuple, set)):
                peaks = list(peaks)
            else:
                peaks = [peaks]

        master.at[j,'Peak positions (s)']  = [fr/el['fps'] for fr in peaks]
        master.at[j,'Peak ISI (s)']  = diff([fr/el['fps'] for fr in peaks])
        
        master.loc[j,'Average ISI (s)']  = mean(diff([fr/el['fps'] for fr in peaks]))
        master.loc[j,'CV'] = std(diff([fr/el['fps'] for fr in peaks]))/mean(diff([fr/el['fps'] for fr in peaks]))
        #master.loc[j, 'Strain'] = el['Mouse ID'][0]
        traceSize = alltraces[el['Cell ID']].dropna().values.shape[0]
        
        master.loc[j,'Average Freq (1/min)']  = 60*len(peaks)/(traceSize/el['fps']) #1/(mean(diff([fr/el['fps'] for fr in peaks])))*60


        master.loc[j,'Number of events'] = len(peaks)

        master.loc[j,'Total duration (s)'] = (traceSize/el['fps'])
        
        amplitudes = list(alltraces[el['Cell ID']].values[peaks])
        master.at[j,'Peak amplitudes'] = amplitudes

        master.loc[j,'Number of cells in recording'] = master[master['Folder']==el['Folder']].shape[0]

        #Cascade part
     #   freq = sum(spikeprob)/durationSpikeProb
     #   maxfreq = max(spikeprob)*el['fps']
     #   totalActiity = sum(spikeprob)
        maxamp = max(amplitudes) if len(amplitudes)>0 else np.nan
     #   master.loc[j,'Average frequency from cascade (Hz)'] = freq
     #   master.loc[j,'Max frequency from cascade (Hz)'] = maxfreq
        master.loc[j,'Max activity dFF0'] = maxamp
     #   master.loc[j,'Total activity from cascade (a.u.)'] = totalActiity
     #   master.loc[j,'Total activity from dff0 (a.u.)'] = sum(amplitudes)
                     

#Calculate peak qualities: we define as "high quality" peaks that:
# 1 - are more than XX SD over noise
# 2 - are longer than xx s:
# A previous definition looked at closeness to a high derivative. 

amplitudeCutoff =2#sd
durationCutoff = 0.2 # s
master['Peak qualities'] = None
master['Peak positions hq'] = None
for j,el in master.iterrows():
    trace = alltraces[el['Cell ID']].dropna()
    traceSize = trace.values.shape[0]
    peaks = el['Peak positions']
    durations =el['Peaks half widths (s)']
    keepPeaks = []
    if not isinstance(peaks, list):
        if isinstance(peaks, np.ndarray):
            peaks = [peaks.item()] if peaks.ndim == 0 else peaks.tolist()
        elif pd.isna(peaks):
            peaks = []
        elif isinstance(peaks, (tuple, set)):
            peaks = list(peaks)
        else:
            peaks = [peaks]
    if not isinstance(durations, list):
        if isinstance(durations, np.ndarray):
            durations = [durations.item()] if durations.ndim == 0 else durations.tolist()
        elif pd.isna(durations):
            durations = []
        elif isinstance(durations, (tuple, set)):
            durations = list(durations)
        else:
            durations = [durations]

    for index,p in enumerate(peaks):
        # keepPeak = False
        # if (trace[p]/el['Noise STD']>= amplitudeCutoff) &  (durations[index]>=durationCutoff):
        #     keepPeak=True

        #AT THE MOMENT, WE KEEP ALL PEAKS, as IT SEEMS BROKEN
        keepPeak = True

        keepPeaks.append(keepPeak)
    master.at[j,'Peak qualities'] = keepPeaks
    master.at[j, 'Peak positions hq'] = np.array(peaks)[keepPeaks]
    if len(peaks)>0:
        master.loc[j,'% kept peaks'] = sum(keepPeaks)/len(peaks)
        master.loc[j,'Kept peaks'] = sum(keepPeaks)
    else:
        master.loc[j,'% kept peaks'] = nan
        master.loc[j,'Kept peaks'] = 0
    


    master.loc[j,'Average Freq hq (1/min)']  = 60*sum(keepPeaks)/(traceSize/el['fps'])
    master.loc[j,'Number of events hq'] = sum(keepPeaks)
    
print('% Peaks of high quality: '+str(master['% kept peaks'].mean()*100))


## Build `masterEventsHq` (event-level table)

This section groups high-quality peaks into shared events per recording.

Rule used here: peaks closer than ~2 seconds are merged into one event. Then events are optionally split when responding cells are separated by large gaps in sequential cell order (`maxSkip`).

Output: one row per event with participating cells, amplitudes, timing offsets, and event type (single vs multiple).

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
import numpy as np

def to_python_list(x):
    a = np.asarray(x)
    if a.ndim == 0:          # scalar / 0-d array
        return [a.item()]    # or just a.item() if you don't want a list
    return a.tolist()

master['Peak positions'] = master['Peak positions'].apply(to_python_list)

master['Peak positions hq'] = master['Peak positions hq'].apply(to_python_list)
CUTOFF = 2#s


#We create a separate dataframe only for HQ events
masterEventsHqList = []
for folder in master['Folder'].unique():
    el = master[master['Folder']==folder]
    el = el[el['Cell type'] == 1]
    #print(el.shape)
    allpeaks = sum(el['Peak positions hq'].values)
    if len(allpeaks)>0:
        allpeaks = sort(list(set(allpeaks))) # remove duplicate events
        distanceThresh = int(CUTOFF*el['fps'].values[0])
        uniqueEvents = [allpeaks[0]]
        for p in allpeaks[1:]:
            if p-uniqueEvents[-1]>distanceThresh:
                uniqueEvents.append(p)

        for event in uniqueEvents:
            
            this_master = pd.Series(dtype=float64)
            
            this_master['Age'] = el['Age'].values[0]
            this_master['Age group'] = el['Age group'].values[0]
            this_master['Folder'] = el['Folder'].values[0]
            this_master['Mouse ID'] = el['Mouse ID'].values[0]
            this_master['Strain'] = el['Strain'].values[0]
            this_master['fps'] = el['fps'].values[0]
            this_master['Frame'] = event
            amplitudes = []
            cells = []
            frameoffset = []
            cellIds = []
            cellResponding = 0
            cellIdsSequential = []
            for k,row in el.iterrows():
                cellResponds = False
                #check if the cell respond by looking at all peaks
                for j in range(0, distanceThresh):
                    if event+j in row['Peak positions hq']:
                        cellResponds = True
                        cellResponding+=1
                        cells.append(int(row['RoiN'].split('_')[1]))
                        cellIds.append(row['Cell ID'])
                        cellIdsSequential.append(row['Cell sequential number'])
                        frameoffset.append(j/row['fps'])
                        amplitudes.append(alltraces[row['Cell ID']].values[event+j-10:event+j+10].mean())
                        break
                if not cellResponds:
                    pass
                    #cells.append(int(row['RoiN'].split('_')[1]))
                    #frameoffset.append(np.nan)
                    #amplitudes.append(alltraces[row['Cell ID']].values[event-10:event+10].mean())
                    
            indices = list(argsort(frameoffset).astype(int))
            this_master['Amplitudes'] = list(array(amplitudes)/1)#array(amplitudes)[indices[0]])
            this_master['Age groups'] = [el['Age group'].values[0]]*len(amplitudes)
            this_master['Ages'] = [el['Age'].values[0]]*len(amplitudes)
            this_master['Offset frame'] = list(array(frameoffset))
            this_master['Cell distances'] = list(abs(array(cellIdsSequential) - array(cellIdsSequential)[indices[0]]))
            this_master['Cell responding'] = cellResponding
            this_master['Total cells'] = el.shape[0]
            this_master['Cell IDs'] = cellIds
            this_master['Sequential cell number'] = cellIdsSequential
            this_master['Sequential cell distances'] = list(diff(cellIdsSequential))
            this_master['Cell responding %'] = cellResponding/el.shape[0]
            masterEventsHqList.append(this_master)
    else:
        print('here')
        print(el['Age'].values[0])

masterEventsHq = pd.DataFrame(masterEventsHqList)
print(masterEventsHq.shape)



#Break any event if cells are far apart more than xx cells:
maxSkip = 4 # max cell skipped to be considered the same event
minCellResponding = 3 # Minimum number of cell responding to be considered a multiple event
masterEventsHqNewList = []
masterEventsHq2 = masterEventsHq.copy()
for j,el in masterEventsHq.iterrows():
  #  print(j)
    indexBreaks = []
    for i,d in enumerate(el['Sequential cell distances']):
        if d>maxSkip:
            indexBreaks.append(i+1)
    if len(indexBreaks)>0:
      #print(indexBreaks)
      #print(el['Sequential cell distances'])
      amplitudes = el['Amplitudes'][:indexBreaks[0]]
      seqCellNumber = el['Sequential cell number'][:indexBreaks[0]]
      ageGroups = el['Age groups'][:indexBreaks[0]]
      offFrame = el['Offset frame'][:indexBreaks[0]]
      celldistances = el['Cell distances'][:indexBreaks[0]]
      cellids = el['Cell IDs'][:indexBreaks[0]]

      seqCellDistances =list(diff(seqCellNumber))

      this_master = el.copy()
      this_master['Amplitudes'] = amplitudes
      this_master['Sequential cell number'] = seqCellNumber
      this_master['Age groups'] = ageGroups
      this_master['Offset frame'] = offFrame
      this_master['Cell distances'] = celldistances
      this_master['Cell IDs'] = cellids
      this_master['Sequential cell distances'] = seqCellDistances
      masterEventsHqNewList.append(this_master)
      for ind,ibreak in enumerate(indexBreaks[:-1]):        
            amplitudes = el['Amplitudes'][indexBreaks[ind]:indexBreaks[ind+1]]
            seqCellNumber = el['Sequential cell number'][indexBreaks[ind]:indexBreaks[ind+1]]
            ageGroups = el['Age groups'][indexBreaks[ind]:indexBreaks[ind+1]]
            offFrame = el['Offset frame'][indexBreaks[ind]:indexBreaks[ind+1]]
            celldistances = el['Cell distances'][indexBreaks[ind]:indexBreaks[ind+1]]
            cellids = el['Cell IDs'][indexBreaks[ind]:indexBreaks[ind+1]]

            seqCellDistances =list(diff(seqCellNumber))

            this_master = el.copy()
            this_master['Amplitudes'] = amplitudes
            this_master['Sequential cell number'] = seqCellNumber
            this_master['Age groups'] = ageGroups
            this_master['Offset frame'] = offFrame
            this_master['Cell distances'] = celldistances
            this_master['Cell IDs'] = cellids
            this_master['Sequential cell distances'] = seqCellDistances
            masterEventsHqNewList.append(this_master)
            amplitudes = el['Amplitudes'][:indexBreaks[0]]

      seqCellNumber = el['Sequential cell number'][indexBreaks[-1]:]
      ageGroups = el['Age groups'][indexBreaks[-1]:]
      offFrame = el['Offset frame'][indexBreaks[-1]:]
      celldistances = el['Cell distances'][indexBreaks[-1]:]
      cellids = el['Cell IDs'][indexBreaks[-1]:]

      seqCellDistances =list(diff(seqCellNumber))

      this_master = el.copy()
      this_master['Amplitudes'] = amplitudes
      this_master['Sequential cell number'] = seqCellNumber
      this_master['Age groups'] = ageGroups
      this_master['Offset frame'] = offFrame
      this_master['Cell distances'] = celldistances
      this_master['Cell IDs'] = cellids
      this_master['Sequential cell distances'] = seqCellDistances
      masterEventsHqNewList.append(this_master)
    
      masterEventsHq.drop(j,axis=0,inplace=True)
masterEventsHq = pd.concat([masterEventsHq, pd.DataFrame(masterEventsHqNewList)], ignore_index=True)

            


print(masterEventsHq.shape)

#Calculate cascade frequency of events, and re-calculate the number and percentage of cell responding based on the break of events

masterEventsHq['Peak amplitudes cellpose (Hz)']  = None
eventCutoff = 3 # if more than 3 cells participate, than it is a multipleEvent
HW = 3 # s window where to find the maximum in the cascade spike trace
for j,el in masterEventsHq.iterrows():
    cellposeamplitudes = []
    for cellid in el['Cell IDs']:
        llimit = el['Frame'] - int(HW*el['fps'])
        rlimit = el['Frame'] + int(HW*el['fps'])
        try:
            cellposeamplitudes.append(allspikeprob[cellid][llimit:rlimit].max()*el['fps'])
        except:
            cellposeamplitudes.append(np.nan)
    masterEventsHq.at[j,'Peak amplitudes cellpose (Hz)'] = cellposeamplitudes
    masterEventsHq.at[j,'Max amplitude cellpose (Hz)'] = max(cellposeamplitudes)  
    masterEventsHq.at[j,'Average amplitude cellpose (Hz)'] = mean(cellposeamplitudes) 

    #Recalc cell responding based on split events
    cellresponding = len(el['Cell IDs'])
    masterEventsHq.loc[j,'Cell responding'] = cellresponding
    masterEventsHq.loc[j,'Cell responding %'] = cellresponding/el['Total cells']
    

    if cellresponding>=minCellResponding:
        masterEventsHq.loc[j,'Event type']='Multiple'
    elif cellresponding==1:
        masterEventsHq.loc[j,'Event type']='Single'
    else:
        masterEventsHq.loc[j,'Event type']='Multiple' # We define them as muliple, these are events with just two cells. Change to undetermined if required


#Calculate the events that have skipping cells
masterEventsHq['Skipping cells'] = False
for j, el in masterEventsHq[masterEventsHq['Cell responding']>=minCellResponding].iterrows():
    cellids2 = el['Sequential cell number']
    dcellids2 = diff(cellids2)
    skippingCells = False
    for i in dcellids2:
        if (abs(i)!=1) and (abs(i)<=maxSkip):
            #print(el['Folder'])
            #print(i)
            
            #print(el['Frame'])
            skippingCells = True
    masterEventsHq.loc[j,'Skipping cells'] = skippingCells
#print(masterEventsHq.loc[masterEventsHq['Cell responding']>minCellResponding,'Skipping cells'].sum())
#print(masterEventsHq.loc[masterEventsHq['Cell responding']>minCellResponding,'Skipping cells'].shape[0])







In [ ]:
# at this point, save the master
master2 = master.copy()
master2['Peak positions'] = [list(joe) for joe in master2['Peak positions']]
master2['Peak positions hq'] = [list(joe) for joe in master2['Peak positions hq']]
master2.to_excel(os.path.join(resultsFOlder,f'{fileHeader}_events_processed.xlsx'))

In [ ]:
for j,el in master.iterrows():
    pp = el['Peak positions']
    if any(diff(pp)<0):
        print('alarm')

## Build peak-level and recording-level summary tables

The next block creates:
- `masterPeak`: one row per peak
- `masterPeakHq`: high-quality subset of peaks
- `masterIndependentCombined`: one row per matched cell across consecutive recordings
- `masterMouse`: per-mouse averages

It also computes single vs multiple event frequencies per cell.

In [ ]:
# Generate a masterPeak where every line is a peak
master['Peaks half widths (s)'] = master['Peaks half widths (s)'].apply(to_python_list)
master['Peaks left limit'] = master['Peaks left limit'].apply(to_python_list)
master['Peaks right limit'] = master['Peaks right limit'].apply(to_python_list)

masterPeak = pd.DataFrame()
HW = 3
for j,el in master.iterrows():
    if el['Cell type'] == 1:
        cellid = el['Cell ID']
        peaks = sort(el['Peak positions'])
        arpeaks = argsort(el['Peak positions'])
        if len(peaks)!= 0:

            #Determine the cascade spikeprob trace peaks
            cellposeamplitudes = []
            for peak in peaks:
                llimit = peak - int(HW*el['fps'])
                rlimit = peak+ int(HW*el['fps'])
                try:
                    cellposeamplitudes.append(allspikeprob[cellid][llimit:rlimit].max()*el['fps'])
                except:
                    cellposeamplitudes.append(np.nan)

            this_masterPeak = pd.DataFrame(
                {
                    'Peak position':peaks,
                    'ISI (s)': hstack([np.nan,diff([fr/el['fps'] for fr in peaks])]),
                    'Half width (s)': [el['Peaks half widths (s)'][nnn] for nnn in arpeaks],
                    'Amplitude':  alltraces[el['Cell ID']].values[el['Peak positions']],
                    'Amplitude cascade (Hz)': cellposeamplitudes,
                    'High quality': [q for q in el['Peak qualities']],
                    'Left limit (s)': [el['Peaks left limit'][nnn]/el['fps'] for nnn in arpeaks],
                    'Right limit (s)': [el['Peaks right limit'][nnn]/el['fps'] for nnn in arpeaks]
                }
            )
        else:
            this_masterPeak = pd.DataFrame({'ISI (s)':[], 'Amplitude':[] , 'Half width (s)':[]})
        
        
        this_masterPeak['Age'] = el['Age']
        this_masterPeak['Age group'] = el['Age group']
        this_masterPeak['Mouse ID'] = el['Mouse ID']
        this_masterPeak['Unique cell ID'] = el['Unique cell ID']
        this_masterPeak['Cell ID'] = el['Cell ID']
        this_masterPeak['Noise STD'] = el['Noise STD']
        this_masterPeak['Folder'] = el['Folder']
        this_masterPeak['Strain'] = el['Strain']
        
        masterPeak = pd.concat((masterPeak,this_masterPeak),axis=0,ignore_index=True)

# Generate a masterPeak where every line is a peak for high quality peaks
masterPeakHq = masterPeak[masterPeak['High quality']]


#Average by cell.
masterPeakHqCells= masterPeakHq.groupby('Unique cell ID').agg({
    'ISI (s)':'mean',
    'Half width (s)':'mean',
    'Amplitude':'mean',
    'Amplitude cascade (Hz)':'mean',
    'Age':'first',
    'Age group':'first',
    'Strain'  :'first',
})



#Calculate separate frequencies for single and multiple events
wrong = 0
for j,el in master.iterrows():
    if el['Cell type']==1:
        eventCutoff = 3 # if events >= this number, they are considered multiple
        cellId = el['Cell ID']
        singleEvents = 0
        multipleEvents = 0 
        for j2,el2 in masterEventsHq[masterEventsHq['Folder']==el['Folder']].iterrows():
            if cellId in el2['Cell IDs']:
                if len(el2['Cell IDs'])== 1:
                    singleEvents+=1
                elif len(el2['Cell IDs'])>= eventCutoff:
                    multipleEvents+=1
        master.loc[j,'Single events N'] = singleEvents
        master.loc[j,'Multiple events N'] = multipleEvents
        master.loc[j,'Orphan events'] =len(el['Peak positions hq']) - (multipleEvents+singleEvents) #orphan events are events that are single in the Peak positions but aggregated as they are too close to each other
        # if singleEvents + multipleEvents != len(el['Peak positions hq']):
        #     wrong+=1
        #     print(wrong)

master['Single events frequency (1/min)'] = 60 * master['Single events N']/master['Total duration (s)']
master['Multiple events frequency (1/min)'] = 60 * master['Multiple events N']/master['Total duration (s)']
master['Single events %'] = master['Single events N']/(master['Single events N']+master['Multiple events N'])
master['Multiple events %'] = master['Multiple events N']/(master['Single events N']+master['Multiple events N'])

In [ ]:
# Generate a master where all consecutive recordings are put together
masterIndependentCombined = master[master['Cell type']==1].groupby('Unique cell ID').agg({'Folder':'sum',
                                                                                          'Independent recordings number':'first',
                                                                                          'Independent recordings ID':'first',
                                                                                          'Mouse ID': 'first',
                                                                                          'Number of events' : 'sum',
                                                                                          'Total duration (s)' : 'sum','Age':'first',
                                                                                          'Single events N':'sum',
                                                                                          'Multiple events N':'sum',
                                                                                          'Age group':'first',
                                                                                          'Kept peaks':'sum',
                                                                                          'Number of cells in recording':'mean',
                                                                                          'Centroid x um':'mean',
                                                                                          'Centroid y um':'mean',
                                                                                          'fps':'first', # we take the first fps
                                                                                          'Strain':'first',
                                                                                          #'Max frequency from cascade (Hz)':'max',
                                                                                          #'Max activity dFF0':'max',
                                                                                          #'Total activity from cascade (a.u.)':'sum',
                                                                                          #'Total activity from dff0 (a.u.)':'sum',

                                                                                          })

masterIndependentCombined['Average Freq (1/min)'] = 60*masterIndependentCombined['Number of events']/masterIndependentCombined['Total duration (s)']
masterIndependentCombined['Average Freq hq (1/min)'] = 60*masterIndependentCombined['Kept peaks']/masterIndependentCombined['Total duration (s)']
masterIndependentCombined['Average single events frequency hq (1/min)'] = 60*masterIndependentCombined['Single events N']/masterIndependentCombined['Total duration (s)']
masterIndependentCombined['Average multiple events frequency hq (1/min)'] = 60*masterIndependentCombined['Multiple events N']/masterIndependentCombined['Total duration (s)']
#masterIndependentCombined['Single events %'] = masterIndependentCombined['Single events N']/(masterIndependentCombined['Kept peaks'])
#masterIndependentCombined['Multiple events %'] = masterIndependentCombined['Multiple events N']/(masterIndependentCombined['Kept peaks'])
masterIndependentCombined['Single events %'] = masterIndependentCombined['Single events N']/(masterIndependentCombined['Single events N'] + masterIndependentCombined['Multiple events N'])
masterIndependentCombined['Multiple events %'] = masterIndependentCombined['Multiple events N']/(masterIndependentCombined['Single events N'] + masterIndependentCombined['Multiple events N'])


#masterIndependentCombined['Average frequency from cascade (Hz)'] = masterIndependentCombined[ 'Total activity from cascade (a.u.)']/masterIndependentCombined['Total duration (s)']



#Generate a master where single mice are averaged
masterMouse = masterIndependentCombined[masterIndependentCombined['Total duration (s)']>300].groupby('Mouse ID').agg({       'Number of events' : 'mean',
                                                                                          'Total duration (s)' : 'first','Age':'first',
                                                                                          'Age group':'first',
                                                                                          'Average Freq (1/min)':'mean',
                                                                                          'Average Freq hq (1/min)':'mean','Strain':'first'}
                                                                                          )





In [ ]:
#masterIndependentCombined[masterIndependentCombined['Total duration (s)']>=300].agg([mean,std,max,min,'count'])[['Average frequency from cascade (Hz)','Max frequency from cascade (Hz)']]

## Quick visualization and exploratory plots

These plots provide a fast overview of frequency, event composition, peak width, and event participation across age and strain groups.

In [ ]:
#sns.catplot(data=master[master['Total duration (s)']>150],x='Age',y='Average Freq hq (1/min)',kind='bar')
data = masterIndependentCombined[#(masterIndependentCombined['Number of events']>3) &
                                 (masterIndependentCombined['Total duration (s)']>300)# &
                                   #(masterIndependentCombined['Number of cells in recording']>10)]
                                  #  (masterIndependentCombined['Average single events frequency hq (1/min)']>0)&
                                   #  (masterIndependentCombined['Average multiple events frequency hq (1/min)']>0)
                                ]   
print(data.shape)
print(masterIndependentCombined.shape)

def point_with_datapoints(data, x, y, hue='Strain', order=None, ylim_vals=None, title=None):
    plt.figure(figsize=(7,5))
    ax = sns.pointplot(
        data=data, x=x, y=y, hue=hue, order=order,
        dodge=False, errorbar='sd'
    )
    sns.stripplot(
        data=data, x=x, y=y, hue=hue, order=order,
        dodge=False, jitter=0.12, alpha=0.5, size=4, ax=ax, legend=False
    )
    if ylim_vals is not None:
        ax.set_ylim(*ylim_vals)
    if title is not None:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

point_with_datapoints(data=data, x='Age', y='Average Freq hq (1/min)', hue='Strain', ylim_vals=(0,2))

age_order = ['First week','Second week','Posthearing']

g = sns.displot(
    data=data,
    x='Average Freq hq (1/min)',
    hue='Strain',
    col='Age group',
    col_wrap=2,
    col_order=age_order,
    kind='hist',
    stat='probability',
    common_norm=False,
    element='step',
    bins=20,
    facet_kws={'sharex': True, 'sharey': True}
 )
g.set_axis_labels('Average Freq hq (1/min)', 'Probability')
g.fig.suptitle('HQ frequency distribution (probability)', y=1.03)
g = sns.displot(
    data=data,
    x='Average Freq hq (1/min)',
    hue='Strain',
    col='Age group',
    col_wrap=2,
    col_order=age_order,
    kind='ecdf',
    facet_kws={'sharex': True, 'sharey': True}
)
g.set_axis_labels('Average Freq hq (1/min)', 'Cumulative probability')
g.fig.suptitle('HQ frequency distribution (cumulative probability)', y=1.03)

point_with_datapoints(
    data=masterMouse, x='Age group', y='Average Freq hq (1/min)', hue='Strain',
    order=['First week','Second week','Posthearing'],
    title='Average Freq hq per mouse (Age group)'
)
point_with_datapoints(
    data=masterMouse, x='Age', y='Average Freq hq (1/min)', hue='Strain',
    title='Average Freq hq per mouse (Age)'
)

point_with_datapoints(data=data, x='Age', y='Average single events frequency hq (1/min)', hue='Strain', ylim_vals=(0,2))

point_with_datapoints(data=data, x='Age', y='Average multiple events frequency hq (1/min)', hue='Strain', ylim_vals=(0,2))

df2 = pd.melt(data.reset_index(),id_vars=['Unique cell ID','Age','Strain'],value_vars=['Average single events frequency hq (1/min)','Average multiple events frequency hq (1/min)'],var_name='Event type',value_name='Frequency (1/min)')
sns.catplot(data=df2,x='Age',y='Frequency (1/min)',hue='Event type',kind='point',col='Strain')
ylim(0,2)
#sns.catplot(data=masterMouse[masterMouse['Total duration (s)']>150],x='Age',y='Average Freq hq (1/min)',kind='bar')
#ylim(

point_with_datapoints(data=data, x='Age', y='Multiple events %', hue='Strain', ylim_vals=(0,1))

point_with_datapoints(data=data, x='Age', y='Single events %', hue='Strain', ylim_vals=(0,1))

sns.relplot(data=data,x='Average single events frequency hq (1/min)',y='Average multiple events frequency hq (1/min)',hue='Strain')

ax = sns.scatterplot(
    data=data,
    x='Average single events frequency hq (1/min)',
    y='Average multiple events frequency hq (1/min)',
    hue='Strain'
)
for strain, d in data.groupby('Strain'):
    sns.regplot(
        data=d,
        x='Average single events frequency hq (1/min)',
        y='Average multiple events frequency hq (1/min)',
        scatter=False,
        ax=ax
    )

#ylim(0,2)
xlim(-.2,1.5)

#sns.catplot(data=data,x='Age',y='Average frequency from cascade (Hz)',kind='point')
#ylim(0)

#sns.catplot(data=data,x='Age',y='Max frequency from cascade (Hz)',kind='point')
#ylim(0)

In [ ]:
sns.displot(data=masterPeakHqCells,x='ISI (s)',hue='Strain',stat='percent',kind='ecdf',col='Age group',col_order = age_order)
xlim(0,200)

In [ ]:
sns.histplot(data=masterPeakHqCells,hue='Age group',x='Half width (s)',binwidth=0.1,stat='percent',common_norm=False)
ylim(0,)
xlim(None,7.5)
sns.catplot(data=masterPeakHqCells,x='Age',y='Half width (s)',kind='point',hue='Strain')


g = sns.catplot(data=masterPeakHqCells,x='Age group',y='Half width (s)')
g.map_dataframe(sns.catplot,data=masterPeakHqCells,x='Age group',y='Half width (s)',kind='point')
#sns.catplot(data=masterPeakHqCells,x='Age group',y='Half width (s)',kind='point',ax=ax)
ylim(0,4)
# sns.jointplot(data=masterPeakHqCells, x="Amplitude cascade (Hz)", y="Half width (s)", hue="Age group",kind='hist')

# sns.jointplot(data=masterPeakHqCells, x="Amplitude", y="Half width (s)", hue="Age group",kind='hist')
# ylim(0,20)

In [ ]:
sns.histplot(data=masterEventsHq[masterEventsHq['Total cells']>=15],x='Cell responding',binwidth=1,cumulative=False,stat='probability',common_norm=False,kde=True,hue='Age group')

In [ ]:
sns.catplot(data=masterEventsHq[masterEventsHq['Cell responding']>=3],x='Age',y='Cell responding',kind='point',hue='Strain')
ylim(0,10)
ylabel('Average number of cells responding')

In [ ]:
df = masterEventsHq.loc[masterEventsHq['Cell responding']>0].groupby(['Folder']).agg({'Skipping cells':'sum','Frame':'count','Age':'first','Mouse ID':'first','Age group':'first','Strain':'first'}).reset_index()
df['Fraction of skipping events'] = df['Skipping cells']/df['Frame']
sns.catplot(data=df,x='Age',y='Fraction of skipping events',kind='point',hue='Strain')
ylim(0,1)
title('Skipping events as a fraction of total events (average per recording)')
df = masterEventsHq.loc[masterEventsHq['Cell responding']>minCellResponding].groupby(['Folder']).agg({'Skipping cells':'sum','Frame':'count','Age':'first','Mouse ID':'first','Age group':'first'}).reset_index()
df['Fraction of skipping events'] = df['Skipping cells']/df['Frame']
sns.catplot(data=df,x='Age',y='Fraction of skipping events',kind='point')
ylim(0,1)
title('Skipping events as a fraction of multicell events (average per recording)')
sns.catplot(data=masterEventsHq[masterEventsHq['Cell responding']>minCellResponding],x='Age',y='Skipping cells',kind='point')
ylim(0,1)
title('Skipping events as a fraction of multicell events (all events pooled)')

sns.catplot(data=masterEventsHq[masterEventsHq['Cell responding']>minCellResponding],x='Age',y='Cell responding',kind='point')
ylim(0,10)
ylabel('Average number of cells responding')


sns.catplot(data=masterEventsHq,hue='Event type',x='Age group',y='Average amplitude cellpose (Hz)',kind='point')
ylim(0)

sns.catplot(data=masterEventsHq,hue='Event type',x='Age group',y='Max amplitude cellpose (Hz)',kind='point')
ylim(0)



## Correlation analysis (cell pairs vs distance)

We compute pairwise correlation coefficients for every unique IHC pair in each field of view and relate correlation to physical distance (in cells and µm).

For downstream export, this analysis is limited to age ≤ P10.

In [ ]:
def distance(P1,P2):
    return sqrt((P1[0]-P2[0])**2 + (P1[1]-P2[1])**2)

In [ ]:
allcorrlist = []
rng = np.random.default_rng()
import tqdm
for j,key in tqdm.tqdm(enumerate(alldff0s.keys()),total=len(alldff0s.keys())):
    dff0 = alldff0s[key].drop('Time (s)',axis=1)
    if dff0.shape[1]<=3:
        continue

    dff0.iloc[:,:] = rollingMedianCorrection(dff0.values,rollingN=2000)
    irn = int(float(key.split('_')[1]))
    min_period = 5*60*master.loc[master['Independent recordings ID']==key,'fps'].values[0]
    
    age = master.loc[master['Independent recordings ID']==key,'Age'].values[0]

    listOFCells = masterIndependentCombined.loc[masterIndependentCombined['Independent recordings ID']==key].index.str.split('_')
    
    #if age<12:
    cellIndexes = [float(loc[2]) for loc in listOFCells]
   # else:
    #    cellIndexes = [float(loc[3]) for loc in listOFCells]
        
    c = dff0.corr(min_periods=int(min_period),method='pearson')
    dff0Scramble = dff0.copy()
    #dff0Scramble.iloc[:,:]= rng.permuted(dff0.values, axis=0)
   # dff0Scramble.iloc[:,:] = rollingMedianCorrection(dff0.values,rollingN=5)
    randmC = dff0Scramble.corr(min_periods=min_period)
    if age<=6:
        ageGroup = 'First week'
    if (age>6):
        ageGroup = 'Second week'
    if age>=12:
        ageGroup = 'Posthearing'
        #else:
     #   age = '5-7'
    for index,i in enumerate(c.index):
        #print(i)
        key1 = key+'_'+str(float(i))
        x1 = masterIndependentCombined.loc[key1,'Centroid x um']
        y1 = masterIndependentCombined.loc[key1,'Centroid y um']   

        for j in c.index[:index]:
            key2 = key+'_'+str(float(j))
            x2 = masterIndependentCombined.loc[key2,'Centroid x um']
            y2 = masterIndependentCombined.loc[key2,'Centroid y um']

            tc = pd.Series(dtype=float64)
            tc['Correlation'] = c.loc[i,j]
            tc['Distance (cells)'] = abs(i-j)
            tc['Distance (um)'] = distance([x1,y1],[x2,y2])
            tc['Age group'] = ageGroup
            tc['Age'] = str(age)
            tc['Recording'] = key
            tc['Mouse ID'] = master.loc[master['Independent recordings ID']==key,'Mouse ID'].values[0]
            tc['Strain'] = master.loc[master['Independent recordings ID']==key,'Strain'].values[0]
            tc['Cell ID'] = key1
            allcorrlist.append(tc)
            #tc = pd.Series()
            #tc['Correlation'] = randmC.loc[i,j]
            #tc['Distance (cells)'] = abs(i-j)
            #tc['Age group'] = 'Noise floor '+ageGroup
            #tc['Age'] = str(age)
            
            #allcorr = allcorr.append(tc,ignore_index=True)
allcorr = pd.DataFrame(allcorrlist)


In [ ]:
def ztrasformAvg(data):
    return tanh(mean(arctanh(data)))

def ztrasformstd(data):
    sd =  tanh(std(arctanh(data)))
    m = ztrasformAvg(data)
    return [m-sd,m+sd]

def ztrasformstdSingle(data):
    sd =  tanh(std(arctanh(data)))
    #m = ztrasformAvg(data)
    return sd

In [ ]:
sns.relplot(data=allcorr,x='Distance (cells)',y='Correlation',kind='line',hue='Strain',col='Age group',estimator = ztrasformAvg,errorbar=ztrasformstd)
xlim(1,30)
#ylim(00,1)

In [ ]:
allcorr['Distance (um) bins'] = pd.cut(allcorr['Distance (um)'],arange(0,200,20),labels=False,ordered=True)

i=pd.cut(allcorr['Distance (um)'],arange(0,200,20))[0]

i.right - i.left 
allcorr['Distance (um) bins']  = allcorr['Distance (um) bins']*(i.right - i.left )
sns.catplot(data=allcorr,x='Distance (um) bins',y='Correlation',kind='point',hue='Strain',col='Age group',aspect=1,estimator=ztrasformAvg,errorbar=ztrasformstd)
xlim(-1,10.5)
ylim(00,1.05)

In [ ]:
allcorr[allcorr['Distance (cells)']<=20].groupby('Age group').count()

## Sample size summary (mouse counts)

Quick checks of how many mice contribute to each age/strain group and related summary counts.

In [ ]:
 display(master.groupby(['Age','Strain'])['Mouse ID'].nunique())

In [ ]:
masterMouse.groupby('Age').count()

In [ ]:
masterEventsHq.groupby(['Age group','Event type'])['Age'].count()/masterEventsHq.shape[0]

In [ ]:
masterIndependentCombined['Single events %'].mean()

In [ ]:
masterIndependentCombined['Multiple events %'].mean()

## Export final analysis tables

Only correlation outputs are filtered to P10 (`Age <= 10`) before export in this section.

In [ ]:
allcorr = allcorr[allcorr['Age']<=10]

In [ ]:
allcorr.to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\allcorr.csv')

In [ ]:
allcorr.groupby(['Age group','Distance (cells)']).agg([ztrasformAvg,ztrasformstdSingle,'count'])['Correlation'].to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\allcorr_avgsByCellDistance.csv')

In [ ]:
masterPeakHqCells.to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\masterPeakHq_averagedPerCell.csv')
masterPeakHq.to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\masterPeakHq.csv')

### Export paths note

The export commands below currently use Windows absolute paths. If you are on Linux/macOS, replace them with valid local output paths first.

In [ ]:
masterIndependentCombined.to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\masterIndependentCombined.csv')
masterIndependentCombined[masterIndependentCombined['Total duration (s)']>300].to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\masterIndependentCombined_longerThan300s.csv')

In [ ]:
masterEventsHq.to_csv('C:\\Users\\LabAdmin\\Documents\\git-repos\\In-Vivo-Analysis\\Data for figures\\IHC activity\\masterEventsHq.csv')

In [ ]:
masterIndependentCombined[masterIndependentCombined['Total duration (s)']>300].mean()

In [ ]:
#sns.catplot(data=masterIndependentCombined[masterIndependentCombined['Total duration (s)']>300],x='Age',y='Average Freq hq (1/min)',kind='swarm')

In [ ]:
# rows = []
# for i,el in masterEventsHq.iterrows():
    
#     peaks = array(el['Amplitudes'])#array(el['Peak amplitudes cellpose (Hz)'])#array(el['Amplitudes'])#
#     if len(peaks)>=2:
#         x = arange(len(peaks))
#         am = argmax(peaks)
#         x = x-x[am]
#         peaks = peaks/max(peaks)
#         #plot(abs(x),peaks)

#         row = pd.DataFrame({'x':abs(x), 'y':peaks})
#         row['Event type'] = len(peaks)
#         rows.append(row)

# df = pd.concat(rows)

In [ ]:
# df['Event type bin'] = pd.cut(df['Event type'],5)

In [ ]:
# sns.catplot(data=df,x='x',y='y',kind='point',hue='Event type bin')

In [ ]:
# sns.catplot(data=df[df['Event type'].isin([2,3,5,10,18])],x='x',y='y',kind='point',hue='Event type')

In [ ]:
# sns.catplot(data=df,x='x',y='y',kind='point',col='Event type',col_wrap=5)